# ARIA v5.0 - Matai'an Three-Act Auditor

This notebook completes the Week 8 class exercise and homework for the 2025 Matai'an Creek barrier-lake event. It uses a reproducible pipeline built on the Microsoft Planetary Computer Data API, with cached AOI crops so the notebook remains rerunnable on this machine.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd()
sys.path.append(str(ROOT))

from week8.aria_v5_pipeline import (
    run_pipeline, fetch_cube, apply_cloud_mask,
    PRE_ITEM_ID, MID_ITEM_ID, POST_ITEM_ID,
    nir_drop, swir_post, bsi_change, ndvi_change,
)

summary = run_pipeline()
summary

## Lab 1 - Scene Selection and TCI Quick-QA

I selected **June 15, 2025** for the Pre scene because it had the cleanest cloud statistics in the window and the Matai'an valley itself remained readable, giving a stable forest baseline before Typhoon Wipha.

I selected **September 11, 2025** for the Mid scene because it coincides with the reported peak lake stage and offers the clearest view of the new turbid-water body in the upper valley.

I selected **October 16, 2025** for the Post scene because cloud cover drops to near-zero while the drained basin and downstream sediment patterns remain fresh enough to audit.

### Candidate Tables

#### PRE

| id                                                     |   cloud_cover | datetime                         |
|:-------------------------------------------------------|--------------:|:---------------------------------|
| S2A_MSIL2A_20250615T023141_R046_T51QUG_20250615T070417 |       8.50106 | 2025-06-15T02:31:41.024000+00:00 |
| S2B_MSIL2A_20250608T022529_R046_T51QUG_20250608T042638 |      35.2512  | 2025-06-08T02:25:29.024000+00:00 |
| S2A_MSIL2A_20250705T023141_R046_T51QUG_20250705T075420 |      36.6418  | 2025-07-05T02:31:41.024000+00:00 |

#### MID

| id                                                     |   cloud_cover | datetime                         |
|:-------------------------------------------------------|--------------:|:---------------------------------|
| S2C_MSIL2A_20250911T022551_R046_T51QUG_20250911T055914 |       13.5193 | 2025-09-11T02:25:51.025000+00:00 |
| S2B_MSIL2A_20250817T022539_R046_T51QUG_20250817T042827 |       17.5858 | 2025-08-17T02:25:39.024000+00:00 |
| S2B_MSIL2A_20250906T022529_R046_T51QUG_20250906T042640 |       28.4602 | 2025-09-06T02:25:29.024000+00:00 |

#### POST

| id                                                     |   cloud_cover | datetime                         |
|:-------------------------------------------------------|--------------:|:---------------------------------|
| S2B_MSIL2A_20251016T022559_R046_T51QUG_20251016T042804 |       2.54306 | 2025-10-16T02:25:59.024000+00:00 |
| S2C_MSIL2A_20251001T022551_R046_T51QUG_20251001T055313 |       6.75612 | 2025-10-01T02:25:51.025000+00:00 |
| S2B_MSIL2A_20251006T022529_R046_T51QUG_20251006T042653 |      13.2761  | 2025-10-06T02:25:29.024000+00:00 |

In [ ]:
candidate_tables = {phase: pd.DataFrame(rows) for phase, rows in summary['candidates'].items()}
for phase, table in candidate_tables.items():
    display(Markdown(f'### {phase.upper()} Top-3 Candidates'))
    display(table)
    display(Image(filename=str(ROOT / 'output' / 'week8_figures' / f'{phase}_candidate_panel.png')))

### Candidate Panels

![PRE candidates](../output/week8_figures/pre_candidate_panel.png)

![MID candidates](../output/week8_figures/mid_candidate_panel.png)

![POST candidates](../output/week8_figures/post_candidate_panel.png)

### Discussion - What the three acts show

- **Act 1 / Pre**: the valley is still forested, with no standing lake visible near the later breach source.
- **Act 2 / Mid**: the Sep 11 scene clearly shows a new turbid-water patch near the verified lake center around `(121.292, 23.696)`.
- **Act 3 / Post**: the lake has drained, while fresh sediment signatures spread across the Guangfu side of the AOI.

![Three-act TCI panel](../output/week8_figures/three_act_tci_panel.png)

## Lab 1 (cont.) - Four Change Metrics

The notebook keeps the required reusable functions in code. I compute them for both **Pre → Mid** and **Pre → Post** so the same logic can support lake birth, source-scar detection, and downstream debris mapping.

In [ ]:
cube_pre = apply_cloud_mask(fetch_cube(PRE_ITEM_ID))
cube_mid = apply_cloud_mask(fetch_cube(MID_ITEM_ID))
cube_post = apply_cloud_mask(fetch_cube(POST_ITEM_ID))

metrics = {
    'nir_drop_pre_mid': nir_drop(cube_pre, cube_mid),
    'swir_post_mid': swir_post(cube_mid),
    'bsi_change_pre_mid': bsi_change(cube_pre, cube_mid),
    'ndvi_change_pre_mid': ndvi_change(cube_pre, cube_mid),
    'nir_drop_pre_post': nir_drop(cube_pre, cube_post),
    'swir_post_post': swir_post(cube_post),
    'bsi_change_pre_post': bsi_change(cube_pre, cube_post),
    'ndvi_change_pre_post': ndvi_change(cube_pre, cube_post),
}
{k: v.shape for k, v in metrics.items()}

In [ ]:
for name in [
    'nir_drop_pre_mid.png', 'swir_post_mid.png', 'bsi_change_pre_mid.png', 'ndvi_change_pre_mid.png',
    'nir_drop_pre_post.png', 'swir_post_post.png', 'bsi_change_pre_post.png', 'ndvi_change_pre_post.png'
]:
    display(Markdown(f'### {name.replace("_", " ").replace(".png", "")}'))
    display(Image(filename=str(ROOT / 'output' / 'week8_figures' / name)))

### Saved change-metric figures

![nir drop pre mid](../output/week8_figures/nir_drop_pre_mid.png)

![swir post mid](../output/week8_figures/swir_post_mid.png)

![bsi change pre mid](../output/week8_figures/bsi_change_pre_mid.png)

![ndvi change pre mid](../output/week8_figures/ndvi_change_pre_mid.png)

![nir drop pre post](../output/week8_figures/nir_drop_pre_post.png)

![swir post post](../output/week8_figures/swir_post_post.png)

![bsi change pre post](../output/week8_figures/bsi_change_pre_post.png)

![ndvi change pre post](../output/week8_figures/ndvi_change_pre_post.png)

## Lab 2 - Detection Masks

### C1. Barrier lake mask

I tested `nir_mid` upper bounds of `0.12`, `0.15`, and `0.18` with the turbid-water rule plus a west-of-`121.33°E` spatial gate. The best area match came from **`nir_mid < 0.18`**, producing a mapped lake area of **0.517 km²**. This is smaller than the NCDR peak benchmark of `0.86 km²`, which suggests the mask remains intentionally conservative after cloud/shadow filtering.

![Barrier lake mask](../output/week8_figures/barrier_lake_mask.png)

### C2. Landslide source scar mask

I built a 10+10 truth set around the upper source area and tuned five threshold pairs using a confusion matrix. The best pair was **nir_drop > 0.1** and **swir_post > 0.2**, with **F1 = 0.947**. I also added an upstream gate (`lon < 121.33`, `lat > 23.68`) so the source-scar layer stays physically tied to the headwall rather than spilling into Guangfu.

In [ ]:
tuning = pd.read_csv(ROOT / 'output' / 'week8_tables' / 'landslide_threshold_tuning.csv', encoding='utf-8-sig')
display(tuning)
display(Image(filename=str(ROOT / 'output' / 'week8_figures' / 'landslide_source_mask.png')))

#### Threshold tuning table

|   nir_drop_min |   swir_post_min |   TP |   FP |   TN |   FN |   precision |   recall |    f1 |
|---------------:|----------------:|-----:|-----:|-----:|-----:|------------:|---------:|------:|
|           0.1  |            0.2  |    9 |    0 |   10 |    1 |           1 |      0.9 | 0.947 |
|           0.15 |            0.25 |    5 |    0 |   10 |    5 |           1 |      0.5 | 0.667 |
|           0.2  |            0.25 |    5 |    0 |   10 |    5 |           1 |      0.5 | 0.667 |
|           0.2  |            0.3  |    3 |    0 |   10 |    7 |           1 |      0.3 | 0.462 |
|           0.15 |            0.3  |    3 |    0 |   10 |    7 |           1 |      0.3 | 0.462 |

![Landslide source mask](../output/week8_figures/landslide_source_mask.png)

### C3. Debris flow footprint mask

The debris rule is different from the landslide rule because the downstream surface is **wet mud over vegetation and paddies**, not bare rock at the source. That means the key signature is a **drop in NDVI** plus a **rise in BSI**, instead of the source-scar pattern of strong NIR loss plus bright SWIR on exposed material.

![Debris flow mask](../output/week8_figures/debris_flow_mask.png)

## Multi-Layer Audit - Eyewitness Impact Table

The impact table combines:

- W3 shelters from the Hualien City range
- W7 top-5 bottlenecks from the Hualien City corridor
- A Week 8 Guangfu overlay with 5 required nodes plus 2 optional nodes

Hit rules follow the assignment: inside for lake/debris, within 200 m for landslide.

In [ ]:
impact = pd.read_csv(ROOT / 'week8' / 'impact_table.csv', encoding='utf-8-sig')
impact.head(15)

#### Impact table preview

| asset                   | asset_name               | type               | location         | W4_terrain_risk   | W7_centrality_rank   | Barrier Lake Hit   | Landslide Hit   | Debris Flow Hit   | Notes                    |
|:------------------------|:-------------------------|:-------------------|:-----------------|:------------------|:---------------------|:-------------------|:----------------|:------------------|:-------------------------|
| Guangfu_Station         | 光復火車站               | W8 Guangfu Overlay | 光復火車站       | —                 | —                    | N                  | N               | Y                 | downstream critical node |
| Guangfu_Township_Office | 光復鄉公所               | W8 Guangfu Overlay | 光復鄉公所       | —                 | —                    | N                  | N               | Y                 | downstream critical node |
| Foxu_Debris_Zone        | 佛祖街沉積區中心         | W8 Guangfu Overlay | 佛祖街沉積區中心 | —                 | —                    | N                  | N               | Y                 | downstream critical node |
| Guangfu_Fire_Station    | 光復消防分隊             | W8 Guangfu Overlay | 光復消防分隊     | —                 | —                    | N                  | N               | Y                 | downstream critical node |
| SH000                   | 主農社區活動中心         | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |
| SH001                   | 中正國小                 | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |
| SH002                   | 中原國小文中三國中預定地 | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |
| SH003                   | 花蓮城隍廟香客大樓       | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |
| SH004                   | 忠孝國小                 | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |
| SH005                   | 主權社區活動中心         | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |
| SH006                   | 中正體育館               | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |
| SH007                   | 國風國中                 | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |
| SH008                   | 明義國小                 | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |
| SH009                   | 花蓮市公所               | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |
| SH010                   | 國立花蓮農業職業學校     | W3 Shelter         | Hualien City     | LOW               | —                    | N                  | N               | N                 | outside event area       |

### Coverage Gap Analysis

- **W3 shelters hit**: 0
- **W7 bottlenecks hit**: 0
- **Guangfu overlay nodes hit by debris**: 4

This is the main teaching point of Week 8: the legacy ARIA coverage was still concentrated north around Hualien City, so it missed the southward operational exposure in Guangfu. ARIA v5.0 fixes that by adding a dedicated local overlay and verifying the event with actual post-scene evidence instead of only upstream risk proxies.

![Final impact map](../output/week8_figures/final_impact_map.png)

## Bonus - AI Advisor Operational Brief

I generated the brief from the impact table and the three-act summary. The final text is reproduced below.

### Operational Brief

Chief of Operations Brief: Sentinel-2 confirms a three-act disaster sequence over Matai'an Creek. The June 15, 2025 pre-scene shows an intact forested valley; the September 11, 2025 mid-scene shows a turbid barrier lake of about 0.517 km² near the verified lake center; and the October 16, 2025 post-scene shows the drained lake basin, an upstream landslide scar of 3.423 km², and a downstream debris footprint of 8.537 km² extending into Guangfu. If ARIA v5.0 had been operational between July 21 and September 23, the clean revisits in early and mid-September could have triggered a persistent-lake warning, flagged the blocked valley, and elevated the bridge and township corridor for UAV confirmation. The coverage-gap result is decisive: the Hualien City shelter and bottleneck layers stay outside the impact zone, while Guangfu overlay nodes carry the post-breach hits. For the next 24 hours, prioritize Highway 9 clearance, station-area access, school resupply, and UAV mapping of the source scar; 4 audited assets already intersect the debris footprint. Before the next barrier-lake event, ARIA should add a standing south-Hualien critical-node layer and an automated lake-growth alert built from recurring optical or SAR scenes.

## Professional Standards Check

- `.env` updated with Week 8 STAC settings and reproducible item IDs
- `mataian_detections.gpkg` written with `barrier_lake`, `landslide_source`, and `debris_flow` layers
- `impact_table.csv` exported
- `output/` includes candidate panels, three-act panel, metric maps, masks, and final impact map
- `README.md` includes chosen scene IDs, coverage gap discussion, and AI diagnostic log